# AG16 - n8n Workflow Integration

**Goal:** Connect AI agents with n8n workflow automation

In this lesson you'll learn how to integrate your AI agents with n8n, enabling them to:
- Trigger automated workflows
- Orchestrate complex multi-step processes
- Integrate with 300+ external services (via n8n nodes)
- Monitor and manage workflow executions

## Why n8n + AI Agents?

**n8n** is a workflow automation platform that connects different services and APIs. When combined with AI agents:

- **Agents decide WHAT to do** (reasoning, planning)
- **n8n handles HOW to do it** (reliable execution, error handling, retries)

### Example Use Cases

1. **Customer Support Agent** → triggers n8n workflow to:
   - Create ticket in Jira
   - Send notification to Slack
   - Update CRM

2. **Research Agent** → triggers n8n workflow to:
   - Scrape data from multiple sources
   - Store in database
   - Generate report and email it

3. **Data Processing Agent** → triggers n8n workflow to:
   - Transform data formats
   - Upload to S3
   - Trigger downstream pipelines

---

## Setup

### 1. Install n8n (if not already running)

```bash
# Using Docker (recommended)
docker run -it --rm \
  --name n8n \
  -p 5678:5678 \
  -v ~/.n8n:/home/node/.n8n \
  n8nio/n8n

# OR using npm
npm install -g n8n
n8n start
```

### 2. Configure n8n API Access

1. Open n8n UI: http://localhost:5678
2. Go to Settings → API
3. Generate an API key
4. Set environment variable:

```bash
export N8N_API_KEY="your-api-key-here"
export N8N_BASE_URL="http://localhost:5678"
```

In [ ]:
import sys
import os
from pathlib import Path

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

# Check configuration
n8n_url = os.getenv("N8N_BASE_URL", "http://localhost:5678")
n8n_key = os.getenv("N8N_API_KEY", "")

print(f"n8n URL: {n8n_url}")
print(f"API Key configured: {'✅' if n8n_key else '❌ Missing - set N8N_API_KEY'}")

if not n8n_key:
    print("\n⚠️  Set your API key:")
    print('   os.environ["N8N_API_KEY"] = "your-key-here"')

---

## Part 1: Basic n8n Client Usage

First, let's test the n8n client directly (without an agent).

In [ ]:
from agentkit.n8n_tools import get_n8n_client

# Initialize client
client = get_n8n_client()

# List available workflows
workflows = client.list_workflows()

print(f"Found {len(workflows)} workflows:")
for wf in workflows[:5]:  # Show first 5
    print(f"  - {wf['name']} (ID: {wf['id']}, Active: {wf.get('active', False)})")

### Create a Test Workflow in n8n

Before continuing, create a simple test workflow in n8n:

1. Open n8n UI: http://localhost:5678
2. Create new workflow
3. Add a **Webhook** node (trigger)
   - Set path to `test-agent`
   - HTTP Method: POST
4. Add a **Set** node
   - Add field: `result = "Success from n8n!"`
   - Add field: `received_data = {{$json.body}}`
5. Connect nodes and **Activate** workflow
6. Copy the workflow ID from the URL

In [ ]:
# Test triggering a workflow
# Replace with your workflow ID
WORKFLOW_ID = "your-workflow-id-here"

result = client.trigger_workflow(
    WORKFLOW_ID,
    data={
        "message": "Hello from agent!",
        "timestamp": "2024-01-01T00:00:00Z"
    }
)

print("Workflow execution result:")
print(result)

---

## Part 2: Agent with n8n Tools

Now let's give an agent the ability to interact with n8n workflows.

In [ ]:
from agentkit.llm import Claude
from agentkit.tools import ToolBox
from agentkit.agent import run_agent
from agentkit.n8n_tools import create_n8n_toolbox

# Initialize LLM
llm = Claude("claude-3-5-sonnet-20241022")

# Create toolbox with n8n tools
toolbox = ToolBox(*create_n8n_toolbox())

# System prompt
system = """
You are an AI agent with access to n8n workflow automation.

You can:
- List available workflows
- Trigger workflows with data
- Check execution status
- Monitor recent executions

Always check what workflows are available before triggering them.
When triggering a workflow, provide meaningful data in JSON format.
""".strip()

# Test task
task = "List all active n8n workflows and tell me what you find."

result = run_agent(llm, toolbox, task, system=system, max_turns=5)

print("\n=== Agent Response ===")
print(result.final_text)
print(f"\nCompleted in {result.turns} turns")

---

## Part 3: Real-World Example - Data Processing Agent

Let's build an agent that can process user requests by orchestrating n8n workflows.

In [ ]:
# Create a more capable agent with additional context
system = """
You are a workflow automation assistant powered by n8n.

When a user asks you to automate something:
1. First, check what workflows are available using n8n_list_workflows
2. Look for a workflow that matches their request
3. If found, get its details to understand what data it needs
4. Trigger the workflow with appropriate data
5. Check the execution status if needed

Be helpful and explain what you're doing at each step.
""".strip()

# Example user request
task = """
I need to process some customer data. 
Check if there's a data processing workflow available and run it with this data:
{
  "customer_id": "12345",
  "email": "test@example.com",
  "action": "update_profile"
}
"""

result = run_agent(llm, toolbox, task, system=system, max_turns=10)

print("\n=== Agent Response ===")
print(result.final_text)
print(f"\nCompleted in {result.turns} turns")

# Show the trace
print("\n=== Execution Trace ===")
for entry in result.tracer.entries:
    if entry["type"] == "tool":
        print(f"Tool: {entry['name']} - Duration: {entry['elapsed']:.2f}s")

---

## Part 4: Advanced Patterns

### Pattern 1: Workflow Chaining

Agent triggers multiple workflows in sequence.

In [ ]:
task = """
I need to:
1. Extract data from our database (use 'data-extraction' workflow)
2. Transform and clean it (use 'data-transformation' workflow)
3. Generate a report (use 'report-generation' workflow)

Execute these in order and pass the output of each step to the next.
The initial data should be: {"table": "users", "date_range": "last_7_days"}
"""

result = run_agent(llm, toolbox, task, system=system, max_turns=15)

print("\n=== Workflow Chain Result ===")
print(result.final_text)

### Pattern 2: Conditional Workflows

Agent decides which workflow to trigger based on input data.

In [ ]:
system = """
You are a smart routing agent for workflow automation.

Based on the user's request type, you should:
- For "urgent" requests: trigger the priority-processing workflow
- For "standard" requests: trigger the normal-processing workflow  
- For "bulk" requests: trigger the batch-processing workflow

Always include the priority level and request type in the data you send.
""".strip()

tasks = [
    "Process this URGENT customer complaint: {...}",
    "Handle this standard inquiry: {...}",
    "Process these 100 records in bulk: [...]",
]

for i, task in enumerate(tasks, 1):
    print(f"\n=== Request {i} ===")
    result = run_agent(llm, toolbox, task, system=system, max_turns=5)
    print(result.final_text)

### Pattern 3: Monitoring & Error Handling

Agent monitors workflow execution and handles failures.

In [ ]:
system = """
You are a workflow monitoring agent.

Your job is to:
1. Check recent workflow executions
2. Identify any failures
3. Report on the status
4. Suggest actions if needed

Be concise but informative.
""".strip()

task = """
Check the last 10 workflow executions.
Tell me if there are any failures and what workflows they're from.
"""

result = run_agent(llm, toolbox, task, system=system, max_turns=5)

print("\n=== Monitoring Report ===")
print(result.final_text)

---

## Part 5: Building a Complete n8n-Powered Agent

Let's combine everything into a production-ready agent.

In [ ]:
from agentkit.trace import Tracer
from agentkit.tools import Tool, tool, ToolBox
from agentkit.n8n_tools import create_n8n_toolbox
import json

# Add custom tools specific to your domain
@tool
def validate_data_format(data: str) -> str:
    """Validate that data is in correct JSON format before sending to n8n.
    
    Args:
        data: JSON string to validate
    
    Returns:
        Validation result message
    """
    try:
        parsed = json.loads(data)
        return f"✅ Valid JSON with {len(parsed)} fields"
    except json.JSONDecodeError as e:
        return f"❌ Invalid JSON: {e}"

@tool
def get_workflow_recommendations(task_description: str) -> str:
    """Get recommended workflows based on task description.
    
    Args:
        task_description: Description of what needs to be done
    
    Returns:
        Recommended workflow names
    """
    # In production, this could query a database or use embeddings
    recommendations = {
        "data": ["data-extraction", "data-transformation"],
        "report": ["report-generation", "analytics-pipeline"],
        "notification": ["email-sender", "slack-notifier"],
        "customer": ["crm-sync", "support-ticket-creator"],
    }
    
    task_lower = task_description.lower()
    matches = []
    for keyword, workflows in recommendations.items():
        if keyword in task_lower:
            matches.extend(workflows)
    
    return json.dumps({
        "recommended_workflows": matches or ["No specific recommendations"],
        "suggestion": "List all workflows to see what's available"
    })

# Create comprehensive toolbox
all_tools = create_n8n_toolbox() + [
    validate_data_format,
    get_workflow_recommendations,
]

toolbox = ToolBox(*all_tools)

# Production system prompt
system = """
You are an intelligent workflow automation agent powered by n8n.

Your capabilities:
- Discover and understand available workflows
- Recommend appropriate workflows for user tasks
- Validate data before triggering workflows
- Execute workflows with proper error handling
- Monitor and report on execution status

Best practices:
1. Always validate data format before triggering workflows
2. Get workflow recommendations when the task is unclear
3. Check workflow details to understand required parameters
4. Provide clear explanations of what you're doing
5. Report both successes and failures clearly

Be helpful, efficient, and reliable.
""".strip()

print("🤖 n8n Workflow Agent Ready")
print(f"   Available tools: {len(all_tools)}")
print("   - 5 n8n tools")
print("   - 2 helper tools")

In [ ]:
# Test the complete agent
test_tasks = [
    "What workflows do you have available?",
    "I need to send a notification to Slack about a new customer signup",
    "Can you help me process customer data and generate a report?",
]

for i, task in enumerate(test_tasks, 1):
    print(f"\n{'='*60}")
    print(f"Task {i}: {task}")
    print('='*60)
    
    tracer = Tracer(llm.model)
    result = run_agent(
        llm, toolbox, task,
        system=system,
        max_turns=10,
        tracer=tracer
    )
    
    print("\n📝 Response:")
    print(result.final_text)
    print(f"\n📊 Stats: {result.turns} turns, {len(tracer.entries)} operations")

---

## Part 6: Exercises

### Exercise 1: Build a Custom Integration

Create an n8n workflow in the UI that:
1. Receives data from your agent
2. Processes it (transform, enrich, etc.)
3. Sends it to an external service (Slack, email, database)

Then build an agent that can trigger it intelligently.

### Exercise 2: Error Recovery Agent

Build an agent that:
1. Monitors workflow executions
2. Detects failures
3. Attempts to re-trigger failed workflows
4. Reports on recovery status

### Exercise 3: Multi-Workflow Orchestration

Create an agent that orchestrates 3+ workflows:
1. Extract data from source A
2. Transform and validate it
3. Load into destination B
4. Send completion notification

Handle the case where any step fails.

### Exercise 4: Dynamic Workflow Selection

Build an agent that:
1. Analyzes the user's natural language request
2. Maps it to available workflows
3. Extracts the required parameters from the request
4. Triggers the appropriate workflow(s)

---

## Summary

You've learned:

✅ **n8n Integration Basics**
  - Setup and configuration
  - Client initialization
  - API authentication

✅ **Agent-Workflow Communication**
  - Listing and discovering workflows
  - Triggering executions with data
  - Monitoring status and results

✅ **Advanced Patterns**
  - Workflow chaining
  - Conditional execution
  - Error handling and recovery
  - Intelligent routing

✅ **Production Considerations**
  - Data validation
  - Workflow recommendations
  - Comprehensive error handling
  - Observability and tracing

### Key Takeaways

1. **Separation of Concerns**: Agents handle reasoning, n8n handles execution
2. **Reliability**: n8n provides robust workflow execution with retries and error handling
3. **Extensibility**: 300+ n8n nodes = 300+ services your agent can orchestrate
4. **Flexibility**: Agents can dynamically choose and configure workflows

### Next Steps

- Explore n8n's 300+ integration nodes
- Build domain-specific workflow libraries
- Implement webhook triggers for async workflows
- Add human-in-the-loop approval for critical workflows
- Create workflow templates that agents can instantiate

---

**Resources:**
- [n8n Documentation](https://docs.n8n.io/)
- [n8n API Reference](https://docs.n8n.io/api/)
- [n8n Community](https://community.n8n.io/)